# Fase 7 — Cascada + last_visible + clipping (`_cuda`)

## Finalidad

Notebook **autocontenido** (como Fases 1–6): entrena **binario + multiclase** desde el baseline V3 vivo y, a continuación, el estimador **`last_visible_idx`** y las comparaciones de **clipping**.

No depende de ejecutar otros notebooks `mejorafase*`. Tras adoptar mejoras previas, este script se regenera desde `train_spine_cascade_binary_to_thoracolumbar_v3.ipynb`.

## Mapa de cambios

| Bloque | Qué hace |
|--------|----------|
| Secciones 1–7 (base) | Mismo pipeline cascada; `OUTPUT_DIR` → `training_runs_cascade_v3_fase7_lastvis_cuda`; checkpoints `binary_spine_cascade_fase7_lastvis_cuda_best.pt` / multiclase fase7. |
| **Sección 8 (Fase 7)** | Target `last_visible`, prep con cascada **de este run**, entrena `LastVisibleEstimator`, clipping y CSV en `training_runs_last_visible_fase7_cuda`. |

**Variante:** CUDA (perfil completo) — Cascada alineada al vivo + last_visible (referencia Colab).

### Colab (`_cuda`)

Montar Drive y ejecutar el notebook completo (cascada + last_visible). Raíz típica: `/content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation`.


## Colab — raíz por defecto

`/content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation`

---



### [Colab — `_cuda`] Comprobar CUDA

En **Google Colab** elige runtime **T4 GPU** (o superior). En VS Code / Jupyter local esta celda solo comprueba si `torch.cuda` ve una GPU.



In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! GPU device:", torch.cuda.get_device_name(0))
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("CUDA is not available. Please ensure you have selected a GPU runtime.")


### [Colab — `_cuda`] Montar Google Drive

Solo aplica en **Colab**. Fuera de Colab se captura `ImportError` y se imprime un mensaje; no hace falta montar Drive si el repo ya esta en disco.



In [ ]:
try:
    from google.colab import drive

    drive.mount('/content/drive')
except ImportError:
    print(
        "[Info] No hay entorno Colab (google.colab): se omite drive.mount. "
        "En VS Code / Jupyter local el repo ya esta en disco; en Colab ejecuta esta celda para montar Drive."
    )


# Entrenamiento en cascada: binary spine -> thoracolumbar

Este notebook construye una **segunda version del pipeline de entrenamiento** con una idea mas cercana al objetivo real del proyecto:

**identificar vertebras**, no solo segmentar la columna completa.

Para eso usamos una cascada de dos etapas:

1. **Modelo 1 - Binary spine**: aprende a separar columna vs fondo.
2. **Modelo 2 - Thoracolumbar multiclass**: recibe una imagen ya enfocada en la region de columna y aprende a distinguir `T1..T12` y `L1..L5`.

La intuicion es sencilla:

- si la red multiclase ve toda la radiografia, desperdicia capacidad en fondo y anatomia irrelevante
- si primero aislamos la columna, la segunda red puede concentrarse en diferencias finas entre vertebras vecinas
- esto nos acerca mas a un pipeline util para identificacion anatomica

## Metodologia de este notebook

La metodologia completa queda asi:

1. Cargar el `manifest` thoracolumbar ya depurado.
2. Hacer un split por paciente/grupo para evitar fuga de informacion:
   - `64% train`
   - `16% val`
   - `20% test`
3. Entrenar una U-Net pequena para **segmentacion binaria**.
4. Usar ese modelo binario para estimar una **ROI espinal** sobre `val` y `test`.
5. Entrenar una segunda U-Net multiclase solo sobre el recorte de columna.
6. Medir el rendimiento final con metricas globales y por vertebra.

Decisiones importantes:

- El target anatomico sigue siendo solo `thoracic + lumbar`.
- Las cervicales excluidas no se convierten en fondo; se tratan como `ignore`.
- En entrenamiento multiclase usamos ROI basada en mascara binaria real para dar estabilidad.
- En validacion y prueba usamos ROI predicha por la red binaria, para aproximarnos mejor al escenario real.

## Arquitectura del modelo

En ambas etapas usamos la misma familia de modelo: una **U-Net pequena**.

Estructura general:

- **Encoder**: extrae caracteristicas cada vez mas abstractas mientras reduce resolucion.
- **Bottleneck**: concentra el contexto global de la imagen.
- **Decoder**: recupera resolucion espacial.
- **Skip connections**: combinan detalle fino del encoder con contexto del decoder.

Canales base:

- `32 -> 64 -> 128 -> 256 -> 512`

Diferencia entre las dos etapas:

- en binaria, la capa final produce `1` canal
- en multiclase, la capa final produce `18` canales:
  - `background`
  - `T1..T12`
  - `L1..L5`

Por que esta arquitectura tiene sentido aqui:

- la U-Net funciona bien en segmentacion medica
- preserva detalle espacial importante
- es lo bastante liviana para iterar rapido
- nos sirve como baseline antes de pasar a arquitecturas mas pesadas

## Antes de ejecutar

Asegurate de tener:

- `Scoliosis_Dataset_V3/Scoliosis_Dataset/indice_dataset.csv`
- `Scoliosis_Dataset_V3/Scoliosis_Dataset/diccionario_etiquetas_T1_T12_L1_L5.json`
- `analysis_outputs_v3/training_manifest_thoracolumbar_v3.csv`

Si faltan librerias:

```python
%pip install -r requirements-notebook.txt
%pip install torch torchvision torchaudio
```

Salidas esperadas:

- pesos en `models/`
- metricas y particiones en `analysis_outputs_v3/training_runs_cascade_v3_fase7_lastvis_cuda/`

In [ ]:
from __future__ import annotations

import copy
import json
import os
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reproducibilidad en GPU (opcional; algo más lento). Útil para acercar runs entre máquinas.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _resolve_maia_project_root(marker: Path) -> Path:
    """Raiz del repo (carpeta que contiene marker). Local, subcarpetas, Colab+Drive."""
    env = os.environ.get('MAIA_PROJECT_ROOT', '').strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / marker).exists():
            return p
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / marker).exists():
            return cand
    # [Colab _cuda] Drive sincronizado: Other computers / Mi portatil / ScoliosisSegmentation
    _colab_cuda_root = Path(r'/content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation')
    if _colab_cuda_root.is_dir() and (_colab_cuda_root / marker).exists():
        return _colab_cuda_root.resolve()
    drive_nb = Path('/content/drive/MyDrive/Colab Notebooks')
    if drive_nb.is_dir():
        direct = drive_nb / 'MAIA-PROYECTO'
        if (direct / marker).exists():
            return direct.resolve()
        for child in drive_nb.iterdir():
            try:
                if child.is_dir() and (child / marker).exists():
                    return child.resolve()
            except OSError:
                continue
    drive_maia = Path('/content/drive/MyDrive/MAIA-PROYECTO')
    if (drive_maia / marker).exists():
        return drive_maia.resolve()
    raise FileNotFoundError(
        f"No se encontro {marker.as_posix()}. Opciones: (1) %cd a la raiz del repo; "
        f"(2) os.environ['MAIA_PROJECT_ROOT'] = r'.../MAIA-PROYECTO'. cwd={cwd}"
    )


ROOT = _resolve_maia_project_root(Path('data') / 'Scoliosis_Dataset')

DATASET_DIR = ROOT / 'data' / 'Scoliosis_Dataset'
DATASET_INDEX_PATH = DATASET_DIR / 'indice_dataset.csv'
LABELS_DICT_PATH = DATASET_DIR / 'diccionario_etiquetas_T1_T12_L1_L5.json'
MANIFEST_PATH = ROOT / 'outputs' / 'analysis_outputs_v3' / 'training_manifest_thoracolumbar_v3.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'analysis_outputs_v3' / 'training_runs_cascade_v3_fase7_lastvis_cuda'
MODEL_DIR = ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (512, 256)
BATCH_SIZE = 4
NUM_WORKERS = 0
LR = 1e-3
WEIGHT_DECAY = 1e-4
BINARY_EPOCHS = 14
MULTICLASS_EPOCHS = 24

# --- [FASE 7] Estimador last_visible + clipping (salidas en LAST_OUTPUT_DIR) ---
LAST_OUTPUT_DIR = ROOT / 'outputs' / 'analysis_outputs_v3' / 'training_runs_last_visible_fase7_cuda'
LAST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE_LAST = (384, 192)
LAST_BATCH_SIZE = 8
LAST_NUM_WORKERS = 0
LAST_EPOCHS = 40
LAST_LR = 6e-4
LAST_WEIGHT_DECAY = 1e-4
LAST_PATIENCE = 8
LAST_DROPOUT = 0.25
LAST_LABEL_SMOOTHING = 0.04
LAST_EXPECTATION_LOSS_WEIGHT = 0.20
LAST_GRAD_CLIP_NORM = 1.0
LAST_HEURISTIC_BLEND = 0.20
LAST_EXPECTATION_BLEND = 0.30
USE_MULTICLASS_TTA = True
USE_AMP = DEVICE.type == "cuda"
PRESENCE_THRESHOLD_PIXELS = 40
PROFILE_BINS = 24
N_VIS_SAMPLES = 8
LAST_VISIBLE_MODEL_PATH = MODEL_DIR / 'last_visible_estimator_fase7_lastvis_cuda_best.pt'
# Comparacion opcional con estimador visible-range del notebook 07 (si existe en disco)
PREV_RANGE_TEST_PATH = (
    ROOT / 'reports' / 'analysis_outputs' / 'visible_range_estimator_thoracolumbar_explained'
    / 'visible_range_test_predictions.csv'
)
TEST_RATIO = 0.20
VAL_RATIO_WITHIN_DEV = 0.20
IGNORE_INDEX = 255
MULTICLASS_SUBSET = 'core'   # cambiar a 'partial' en una iteracion posterior
TARGET_SUBSET = MULTICLASS_SUBSET  # alias notebook 07
   # cambiar a 'partial' en una iteracion posterior
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 18
ROI_PAD_Y = 30
ROI_JITTER_X = 10
ROI_JITTER_Y = 14
MIN_FOREGROUND_PIXELS = 24

TARGET_LABELS = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
EXCLUDED_LABELS = ['C7', 'C6', 'C5', 'C4', 'C3']

dataset_index_df = pd.read_csv(DATASET_INDEX_PATH)
# Normalizar esquema del indice V3 para compatibilidad con el pipeline.
dataset_index_df = dataset_index_df.rename(columns={
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_path',
})
dataset_index_df['mask_path'] = dataset_index_df['multiclass_path']
manifest_df = pd.read_csv(MANIFEST_PATH)
with open(LABELS_DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

multiclass_key = 'multiclass_id_png' if 'multiclass_id_png' in labels_dict else 'mascara_multiclase_id_png'
multiclass_map = {int(k): v for k, v in labels_dict[multiclass_key].items()}
label_to_id = {label: class_id for class_id, label in multiclass_map.items()}
target_id_to_train_id = {label_to_id[label]: idx + 1 for idx, label in enumerate(TARGET_LABELS)}
class_names = ['background'] + TARGET_LABELS
num_classes = len(class_names)

join_cols = ['split', 'image', 'patient_id', 'radiograph_path']
dataset_subset = dataset_index_df[join_cols + ['label_binary_path', 'multiclass_path']].copy()
train_table = manifest_df.merge(dataset_subset, on=join_cols, how='left', suffixes=('', '_idx'))

train_table['multiclass_path'] = train_table['mask_path'].fillna(train_table['multiclass_path'])
train_table['radiograph_path_abs'] = train_table['radiograph_path'].apply(lambda rel: str((DATASET_DIR / rel).resolve()))
train_table['binary_mask_path_abs'] = train_table['label_binary_path'].apply(lambda rel: str((DATASET_DIR / rel).resolve()))
train_table['multiclass_mask_path_abs'] = train_table['multiclass_path'].apply(lambda rel: str((DATASET_DIR / rel).resolve()))

flag_cols = [
    'usable_for_binary_spine',
    'usable_for_thoracolumbar_core',
    'usable_for_thoracolumbar_partial',
    'needs_annotation_review',
    'usable_for_cobb_regression',
]
for col in flag_cols:
    if col in train_table.columns:
        train_table[col] = train_table[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == 'true'
        )

print('FASE 7 cascada+lastvis (cuda) — OUTPUT_DIR:', OUTPUT_DIR)
print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('Muestras totales del manifest:', len(train_table))
print('Subset binario utilizable:', int(train_table['usable_for_binary_spine'].sum()))
print('Subset thoracolumbar core:', int(train_table['usable_for_thoracolumbar_core'].sum()))
print('Subset thoracolumbar partial:', int(train_table['usable_for_thoracolumbar_partial'].sum()))
print('Casos sugeridos para revision manual:', int(train_table['needs_annotation_review'].sum()))

display(
    train_table[
        [
            'unique_sample_id', 'split', 'image', 'group_id_for_split',
            'num_visible_target_vertebrae', 'visible_target_span_signature',
            'excluded_cervical_count', 'total_internal_missing_count',
            'unexpected_label_count', 'usable_for_binary_spine',
            'usable_for_thoracolumbar_core', 'usable_for_thoracolumbar_partial',
            'needs_annotation_review'
        ]
    ].head()
)


## Seccion 1. Particion de datos

Aqui hacemos el split del dataset con dos objetivos:

1. reservar un `test` real que no se use para decisiones del modelo
2. tener un `validation` intermedio para elegir el mejor checkpoint

Es importante que el split sea por `group_id_for_split`, porque asi evitamos que imagenes relacionadas queden repartidas entre entrenamiento y evaluacion.

In [ ]:
def make_group_train_val_test_split(
    df_in: pd.DataFrame,
    test_ratio: float = 0.20,
    val_ratio_within_dev: float = 0.20,
    seed: int = 42,
) -> pd.DataFrame:
    work = df_in.reset_index(drop=True).copy()

    holdout_splitter = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=seed)
    dev_idx, test_idx = next(holdout_splitter.split(work, groups=work['group_id_for_split']))
    dev_df = work.iloc[dev_idx].copy()
    test_df = work.iloc[test_idx].copy()

    val_splitter = GroupShuffleSplit(n_splits=1, test_size=val_ratio_within_dev, random_state=seed)
    train_idx, val_idx = next(val_splitter.split(dev_df, groups=dev_df['group_id_for_split']))
    train_df = dev_df.iloc[train_idx].copy()
    val_df = dev_df.iloc[val_idx].copy()

    train_df['partition'] = 'train'
    val_df['partition'] = 'val'
    test_df['partition'] = 'test'
    return pd.concat([train_df, val_df, test_df], ignore_index=True)


binary_df = train_table.loc[train_table['usable_for_binary_spine']].copy()
multiclass_flag = 'usable_for_thoracolumbar_core' if MULTICLASS_SUBSET == 'core' else 'usable_for_thoracolumbar_partial'
multiclass_df = train_table.loc[train_table[multiclass_flag] & ~train_table['needs_annotation_review']].copy()

binary_splits_df = make_group_train_val_test_split(
    binary_df,
    test_ratio=TEST_RATIO,
    val_ratio_within_dev=VAL_RATIO_WITHIN_DEV,
    seed=SEED,
)
multiclass_splits_df = make_group_train_val_test_split(
    multiclass_df,
    test_ratio=TEST_RATIO,
    val_ratio_within_dev=VAL_RATIO_WITHIN_DEV,
    seed=SEED,
)

binary_split_path = OUTPUT_DIR / 'binary_spine_split_train_val_test.csv'
multiclass_split_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_split_train_val_test.csv'
binary_splits_df.to_csv(binary_split_path, index=False)
multiclass_splits_df.to_csv(multiclass_split_path, index=False)

print('Split binario guardado en:', binary_split_path)
print('Split multiclase guardado en:', multiclass_split_path)

print('\nDistribucion binaria:')
display(binary_splits_df['partition'].value_counts().rename_axis('partition').reset_index(name='images'))
display(binary_splits_df.groupby(['partition', 'split']).size().rename('images').reset_index())

print('\nDistribucion multiclase:')
display(multiclass_splits_df['partition'].value_counts().rename_axis('partition').reset_index(name='images'))
display(multiclass_splits_df.groupby(['partition', 'split']).size().rename('images').reset_index())

for df_check in [binary_splits_df, multiclass_splits_df]:
    train_groups = set(df_check.loc[df_check['partition'] == 'train', 'group_id_for_split'])
    val_groups = set(df_check.loc[df_check['partition'] == 'val', 'group_id_for_split'])
    test_groups = set(df_check.loc[df_check['partition'] == 'test', 'group_id_for_split'])
    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

## Seccion 2. Preparacion de imagenes, mascaras y ROI

Esta es una de las partes mas importantes del notebook.

Que hacemos aqui:

- leemos radiografia y mascaras
- convertimos la mascara binaria a `0/1`
- convertimos la mascara multiclase original al nuevo espacio de clases
- calculamos una **ROI** que encierre la columna
- recortamos la imagen a esa ROI antes de pasarla al modelo multiclase

Idea clave:

- en `train` multiclase usamos ROI basada en la mascara binaria real para dar estabilidad
- en `val/test` multiclase usamos ROI predicha por el modelo binario

Asi la segunda red aprende con una region anatomica coherente, pero al mismo tiempo la evaluamos en un flujo mas parecido al uso real.

In [ ]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_image(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr).resize((size[1], size[0]), resample=Image.BILINEAR))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_binary_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_gray(path)
    mask = (mask >= 127).astype(np.uint8)
    if size is not None:
        mask = resize_mask(mask, size)
    return mask


def build_multiclass_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.int32)
    out = np.zeros_like(raw, dtype=np.uint8)
    out[raw != 0] = IGNORE_INDEX
    for old_id, new_id in target_id_to_train_id.items():
        out[raw == old_id] = new_id
    if size is not None:
        out = resize_mask(out, size)
    return out


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(
    bbox: tuple[int, int, int, int],
    image_shape: tuple[int, int],
    pad_x: int = 18,
    pad_y: int = 30,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def jitter_bbox(
    bbox: tuple[int, int, int, int],
    image_shape: tuple[int, int],
    max_jitter_x: int = 10,
    max_jitter_y: int = 14,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    dx0 = np.random.randint(-max_jitter_x, max_jitter_x + 1)
    dx1 = np.random.randint(-max_jitter_x, max_jitter_x + 1)
    dy0 = np.random.randint(-max_jitter_y, max_jitter_y + 1)
    dy1 = np.random.randint(-max_jitter_y, max_jitter_y + 1)
    return clamp_bbox((x0 + dx0, y0 + dy0, x1 + dx1, y1 + dy1), image_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def scale_bbox(
    bbox: tuple[int, int, int, int],
    src_shape: tuple[int, int],
    dst_shape: tuple[int, int],
) -> tuple[int, int, int, int]:
    src_h, src_w = src_shape
    dst_h, dst_w = dst_shape
    x0, y0, x1, y1 = bbox
    sx = dst_w / src_w
    sy = dst_h / src_h
    scaled = (
        int(round(x0 * sx)),
        int(round(y0 * sy)),
        int(round(x1 * sx)),
        int(round(y1 * sy)),
    )
    return clamp_bbox(scaled, dst_shape)


def full_image_bbox(image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    return 0, 0, w, h


def build_roi_record_from_binary_mask(
    binary_mask: np.ndarray,
    image_shape: tuple[int, int],
    roi_source: str,
    pad_x: int = ROI_PAD_X,
    pad_y: int = ROI_PAD_Y,
) -> dict:
    bbox = bbox_from_mask(binary_mask, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bbox is None:
        bbox = full_image_bbox(image_shape)
        roi_source = f'{roi_source}_fallback_full_image'
    else:
        bbox = expand_bbox(bbox, image_shape=image_shape, pad_x=pad_x, pad_y=pad_y)
    x0, y0, x1, y1 = bbox
    return {
        'bbox_x0': x0,
        'bbox_y0': y0,
        'bbox_x1': x1,
        'bbox_y1': y1,
        'bbox_width': x1 - x0,
        'bbox_height': y1 - y0,
        'roi_source': roi_source,
    }




def apply_fase4_roi_geom_augment_uint8(
    image_crop_u8: np.ndarray,
    multiclass_crop: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Augmentación geométrica suave en crop ROI multiclase (solo train).
    Rotación ±4° y escala 0,98–1,02; 50% de no aplicar nada (radiografías).
    """
    if np.random.random() > 0.5:
        return image_crop_u8, multiclass_crop
    h, w = image_crop_u8.shape[:2]
    if h < 8 or w < 8:
        return image_crop_u8, multiclass_crop

    angle = float(np.random.uniform(-4.0, 4.0))
    scale = float(np.random.uniform(0.98, 1.02))
    nh = max(2, int(round(h * scale)))
    nw = max(2, int(round(w * scale)))

    img_p = Image.fromarray(image_crop_u8, mode="L")
    msk_u8 = multiclass_crop.astype(np.uint8, copy=False)
    msk_p = Image.fromarray(msk_u8, mode="L")
    img_p = img_p.resize((nw, nh), resample=Image.BILINEAR)
    msk_p = msk_p.resize((nw, nh), resample=Image.NEAREST)

    arr_i = np.array(img_p)
    arr_m = np.array(msk_p)
    hi, wi = arr_i.shape
    out_i = np.zeros((h, w), dtype=np.uint8)
    out_m = np.full((h, w), int(IGNORE_INDEX), dtype=np.uint8)
    if hi <= h and wi <= w:
        y0 = (h - hi) // 2
        x0 = (w - wi) // 2
        out_i[y0 : y0 + hi, x0 : x0 + wi] = arr_i
        out_m[y0 : y0 + hi, x0 : x0 + wi] = arr_m
    else:
        y0 = (hi - h) // 2
        x0 = (wi - w) // 2
        out_i = arr_i[y0 : y0 + h, x0 : x0 + w]
        out_m = arr_m[y0 : y0 + h, x0 : x0 + w]

    ri = Image.fromarray(out_i, mode="L")
    rm = Image.fromarray(out_m.astype(np.uint8), mode="L")
    ri2 = ri.rotate(angle, resample=Image.BILINEAR, fillcolor=0, expand=False)
    rm2 = rm.rotate(angle, resample=Image.NEAREST, fillcolor=int(IGNORE_INDEX), expand=False)
    out_i2 = np.array(ri2)
    out_m2 = np.array(rm2).astype(multiclass_crop.dtype, copy=False)
    return out_i2, out_m2


def prepare_multiclass_cascade_sample(
    row: pd.Series,
    output_size: tuple[int, int],
    roi_mode: str,
    roi_lookup: dict | None = None,
    apply_jitter: bool = False,
    apply_roi_augment: bool = False,
) -> dict:
    image_raw = read_gray(row['radiograph_path_abs'])
    binary_raw = build_binary_mask(row['binary_mask_path_abs'], size=None)
    multiclass_raw = build_multiclass_mask(row['multiclass_mask_path_abs'], size=None)
    image_shape = image_raw.shape

    if roi_mode == 'gt_binary':
        roi_meta = build_roi_record_from_binary_mask(binary_raw, image_shape=image_shape, roi_source='gt_binary')
    elif roi_mode == 'pred_binary':
        if roi_lookup is None:
            raise ValueError('roi_lookup es obligatorio para roi_mode=pred_binary')
        roi_meta = roi_lookup.get(row['unique_sample_id'])
        if roi_meta is None:
            roi_meta = build_roi_record_from_binary_mask(
                np.zeros_like(binary_raw),
                image_shape=image_shape,
                roi_source='pred_binary_missing',
            )
    else:
        raise ValueError(f'roi_mode no soportado: {roi_mode}')

    bbox = (int(roi_meta['bbox_x0']), int(roi_meta['bbox_y0']), int(roi_meta['bbox_x1']), int(roi_meta['bbox_y1']))
    if apply_jitter:
        bbox = jitter_bbox(bbox, image_shape=image_shape, max_jitter_x=ROI_JITTER_X, max_jitter_y=ROI_JITTER_Y)

    image_crop = crop_array(image_raw, bbox)
    multiclass_crop = crop_array(multiclass_raw, bbox)

    if apply_roi_augment:
        image_crop, multiclass_crop = apply_fase4_roi_geom_augment_uint8(image_crop, multiclass_crop)

    image_crop = resize_image(image_crop, output_size).astype(np.float32) / 255.0
    multiclass_crop = resize_mask(multiclass_crop, output_size).astype(np.int64)
    image_crop = np.expand_dims(image_crop, axis=0)

    return {
        'image': image_crop,
        'mask': multiclass_crop,
        'bbox': bbox,
        'roi_source': roi_meta['roi_source'],
        'image_shape': image_shape,
    }


class BinarySpineDataset(Dataset):
    def __init__(self, table: pd.DataFrame, image_size: tuple[int, int]):
        self.table = table.reset_index(drop=True).copy()
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str | tuple[int, int]]:
        row = self.table.iloc[idx]
        image_raw = read_gray(row['radiograph_path_abs'])
        image_resized = resize_image(image_raw, self.image_size).astype(np.float32) / 255.0
        image_resized = np.expand_dims(image_resized, axis=0)
        mask = build_binary_mask(row['binary_mask_path_abs'], size=self.image_size).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)
        return {
            'image': torch.tensor(image_resized, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32),
            'sample_id': row['unique_sample_id'],
            'original_shape': tuple(image_raw.shape),
        }


class CascadedThoracolumbarDataset(Dataset):
    def __init__(
        self,
        table: pd.DataFrame,
        image_size: tuple[int, int],
        roi_mode: str,
        roi_lookup: dict | None = None,
        apply_jitter: bool = False,
        apply_roi_augment: bool = False,
    ):
        self.table = table.reset_index(drop=True).copy()
        self.image_size = image_size
        self.roi_mode = roi_mode
        self.roi_lookup = roi_lookup
        self.apply_jitter = apply_jitter
        self.apply_roi_augment = apply_roi_augment

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str]:
        row = self.table.iloc[idx]
        sample = prepare_multiclass_cascade_sample(
            row=row,
            output_size=self.image_size,
            roi_mode=self.roi_mode,
            roi_lookup=self.roi_lookup,
            apply_jitter=self.apply_jitter,
            apply_roi_augment=self.apply_roi_augment,
        )
        return {
            'image': torch.tensor(sample['image'], dtype=torch.float32),
            'mask': torch.tensor(sample['mask'], dtype=torch.int64),
            'sample_id': row['unique_sample_id'],
            'roi_source': sample['roi_source'],
        }


preview_row = multiclass_splits_df.query("partition == 'train'").reset_index(drop=True).iloc[0]
preview_image = read_gray(preview_row['radiograph_path_abs'])
preview_binary = build_binary_mask(preview_row['binary_mask_path_abs'], size=None)
preview_multiclass = build_multiclass_mask(preview_row['multiclass_mask_path_abs'], size=None)
preview_roi = prepare_multiclass_cascade_sample(
    row=preview_row,
    output_size=IMG_SIZE_MULTICLASS,
    roi_mode='gt_binary',
    roi_lookup=None,
    apply_jitter=False,
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(preview_image, cmap='gray')
axes[0, 0].set_title('Radiografia original')
axes[0, 1].imshow(preview_binary, cmap='gray')
axes[0, 1].set_title('Mascara binaria real')
axes[1, 0].imshow(preview_roi['image'][0], cmap='gray')
axes[1, 0].set_title('ROI usada para multiclase')
preview_mask_plot = preview_roi['mask'].copy()
preview_mask_plot[preview_mask_plot == IGNORE_INDEX] = 0
axes[1, 1].imshow(preview_mask_plot, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
axes[1, 1].set_title('Mascara multiclase recortada')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

## Seccion 3. Definicion de la arquitectura y de las metricas

En esta seccion definimos:

- la U-Net pequena
- las funciones de perdida
- las metricas para entrenamiento y evaluacion

La idea es mantener una base clara y reutilizable.

En lugar de optimizar solo por `loss`, guardamos el mejor checkpoint usando:

- `val_dice` en binaria
- `val_macro_dice_fg` en multiclase

Eso es mas coherente con el objetivo real de segmentacion.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UNetSmall(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConv(in_channels, base)
        self.e2 = DoubleConv(base, base * 2)
        self.e3 = DoubleConv(base * 2, base * 4)
        self.e4 = DoubleConv(base * 4, base * 8)
        self.b = DoubleConv(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConv(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConv(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConv(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))

        d4 = self.u4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.d4(d4)

        d3 = self.u3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.d3(d3)

        d2 = self.u2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.d2(d2)

        d1 = self.u1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.d1(d1)
        return self.head(d1)


def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def dice_loss_binary(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (denom + eps)
    return 1.0 - dice.mean()


def dice_loss_multiclass(
    logits: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
    ignore_index: int = 255,
    eps: float = 1e-6,
) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    valid = targets != ignore_index
    safe_targets = targets.clone()
    safe_targets[~valid] = 0
    target_one_hot = F.one_hot(safe_targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
    valid = valid.unsqueeze(1)
    probs = probs * valid
    target_one_hot = target_one_hot * valid

    probs = probs[:, 1:, :, :]
    target_one_hot = target_one_hot[:, 1:, :, :]

    intersection = (probs * target_one_hot).sum(dim=(0, 2, 3))
    denom = probs.sum(dim=(0, 2, 3)) + target_one_hot.sum(dim=(0, 2, 3))
    dice = (2.0 * intersection + eps) / (denom + eps)
    return 1.0 - dice.mean()


def evaluate_binary(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    bce = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    total_batches = 0
    intersection = 0.0
    pred_area = 0.0
    target_area = 0.0
    correct = 0.0
    total_pixels = 0.0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(DEVICE)
            targets = batch['mask'].to(DEVICE)
            logits = model(images)
            loss = bce(logits, targets) + dice_loss_binary(logits, targets)
            total_loss += float(loss.item())
            total_batches += 1

            preds = (torch.sigmoid(logits) >= BINARY_THRESHOLD).float()
            intersection += float((preds * targets).sum().item())
            pred_area += float(preds.sum().item())
            target_area += float(targets.sum().item())
            correct += float((preds == targets).sum().item())
            total_pixels += float(targets.numel())

    union = pred_area + target_area - intersection
    return {
        'loss': total_loss / max(total_batches, 1),
        'dice': (2.0 * intersection + 1e-6) / (pred_area + target_area + 1e-6),
        'iou': (intersection + 1e-6) / (union + 1e-6),
        'pixel_accuracy': (correct + 1e-6) / (total_pixels + 1e-6),
    }


def evaluate_multiclass(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[dict[str, float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    correct = 0.0
    total_valid_pixels = 0.0
    intersection = np.zeros(num_classes, dtype=np.float64)
    pred_area = np.zeros(num_classes, dtype=np.float64)
    target_area = np.zeros(num_classes, dtype=np.float64)

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(DEVICE)
            targets = batch['mask'].to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, targets) + dice_loss_multiclass(logits, targets, num_classes=num_classes, ignore_index=IGNORE_INDEX)
            total_loss += float(loss.item())
            total_batches += 1

            preds = torch.argmax(logits, dim=1)
            valid = targets != IGNORE_INDEX
            correct += float(((preds == targets) & valid).sum().item())
            total_valid_pixels += float(valid.sum().item())

            preds_np = preds[valid].detach().cpu().numpy()
            targets_np = targets[valid].detach().cpu().numpy()
            for class_idx in range(num_classes):
                pred_mask = preds_np == class_idx
                target_mask = targets_np == class_idx
                intersection[class_idx] += np.logical_and(pred_mask, target_mask).sum()
                pred_area[class_idx] += pred_mask.sum()
                target_area[class_idx] += target_mask.sum()

    dice = (2.0 * intersection + 1e-6) / (pred_area + target_area + 1e-6)
    iou = (intersection + 1e-6) / (pred_area + target_area - intersection + 1e-6)
    per_class_df = pd.DataFrame(
        {
            'class_id': np.arange(num_classes),
            'class_name': class_names,
            'pred_pixels': pred_area,
            'target_pixels': target_area,
            'dice': dice,
            'iou': iou,
        }
    )
    fg_df = per_class_df.loc[per_class_df['class_id'] > 0].copy()
    metrics = {
        'loss': total_loss / max(total_batches, 1),
        'pixel_accuracy': (correct + 1e-6) / (total_valid_pixels + 1e-6),
        'macro_dice_fg': float(fg_df['dice'].mean()),
        'macro_iou_fg': float(fg_df['iou'].mean()),
    }
    return metrics, per_class_df


def plot_history(history_df: pd.DataFrame, title: str) -> None:
    metric_cols = [col for col in history_df.columns if col != 'epoch']
    ncols = 2
    nrows = int(math.ceil(len(metric_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, max(4, 4 * nrows)))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, metric_cols):
        ax.plot(history_df['epoch'], history_df[col], marker='o')
        ax.set_title(col)
        ax.set_xlabel('epoch')
        ax.grid(True, alpha=0.3)
    for ax in axes[len(metric_cols):]:
        ax.axis('off')
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


binary_probe = UNetSmall(in_channels=1, out_channels=1)
multiclass_probe = UNetSmall(in_channels=1, out_channels=num_classes)
print('Parametros entrenables binarios:', count_trainable_params(binary_probe))
print('Parametros entrenables multiclase:', count_trainable_params(multiclass_probe))
del binary_probe, multiclass_probe

## Seccion 4. Entrenamiento del modelo binario

Aqui entrenamos la primera etapa de la cascada.

Objetivo:

- detectar la columna como una sola region

Por que esta etapa es importante:

- si el modelo binario delimita bien la columna, la segunda red recibe una entrada mucho mas informativa
- esta etapa funciona como un localizador anatomico grueso

Seleccion del mejor modelo:

- entrenamos por varias epocas
- evaluamos en `val`
- guardamos el checkpoint con mejor `val_dice`

In [ ]:
binary_train_ds = BinarySpineDataset(binary_splits_df.query("partition == 'train'"), image_size=IMG_SIZE_BINARY)
binary_val_ds = BinarySpineDataset(binary_splits_df.query("partition == 'val'"), image_size=IMG_SIZE_BINARY)
binary_test_ds = BinarySpineDataset(binary_splits_df.query("partition == 'test'"), image_size=IMG_SIZE_BINARY)

binary_train_loader = DataLoader(binary_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
binary_val_loader = DataLoader(binary_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
binary_test_loader = DataLoader(binary_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

binary_model = UNetSmall(in_channels=1, out_channels=1).to(DEVICE)
binary_optimizer = torch.optim.AdamW(binary_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
binary_bce = nn.BCEWithLogitsLoss()

binary_history = []
best_binary_state = None
best_binary_val_dice = -1.0
binary_start = time.time()

for epoch in range(1, BINARY_EPOCHS + 1):
    binary_model.train()
    running_loss = 0.0
    for batch in binary_train_loader:
        images = batch['image'].to(DEVICE)
        targets = batch['mask'].to(DEVICE)

        binary_optimizer.zero_grad()
        logits = binary_model(images)
        loss = binary_bce(logits, targets) + dice_loss_binary(logits, targets)
        loss.backward()
        binary_optimizer.step()
        running_loss += float(loss.item())

    train_metrics = evaluate_binary(binary_model, binary_train_loader)
    val_metrics = evaluate_binary(binary_model, binary_val_loader)
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_dice': train_metrics['dice'],
        'train_iou': train_metrics['iou'],
        'train_pixel_accuracy': train_metrics['pixel_accuracy'],
        'val_loss': val_metrics['loss'],
        'val_dice': val_metrics['dice'],
        'val_iou': val_metrics['iou'],
        'val_pixel_accuracy': val_metrics['pixel_accuracy'],
    }
    binary_history.append(row)

    if val_metrics['dice'] > best_binary_val_dice:
        best_binary_val_dice = val_metrics['dice']
        best_binary_state = {k: v.detach().cpu().clone() for k, v in binary_model.state_dict().items()}

    print(
        f"[Binary][Epoch {epoch:02d}/{BINARY_EPOCHS}] "
        f"train_dice={row['train_dice']:.4f} "
        f"val_dice={row['val_dice']:.4f} "
        f"train_iou={row['train_iou']:.4f} "
        f"val_iou={row['val_iou']:.4f}"
    )

binary_history_df = pd.DataFrame(binary_history)
binary_model.load_state_dict(best_binary_state)
binary_test_metrics = evaluate_binary(binary_model, binary_test_loader)
binary_elapsed_min = (time.time() - binary_start) / 60.0

binary_model_path = MODEL_DIR / 'binary_spine_cascade_fase7_lastvis_cuda_best.pt'
binary_history_path = OUTPUT_DIR / 'binary_spine_history.csv'
binary_test_metrics_path = OUTPUT_DIR / 'binary_spine_test_metrics.csv'
torch.save(binary_model.state_dict(), binary_model_path)
binary_history_df.to_csv(binary_history_path, index=False)
pd.DataFrame([binary_test_metrics]).to_csv(binary_test_metrics_path, index=False)

print('\nMejor val_dice binario:', round(best_binary_val_dice, 4))
print('Metricas finales en test:', binary_test_metrics)
print('Tiempo de entrenamiento binario (min):', round(binary_elapsed_min, 2))

display(binary_history_df)
display(pd.DataFrame([binary_test_metrics]))
plot_history(binary_history_df, 'Historia de entrenamiento: binary spine cascada')

## Seccion 5. Construccion de la ROI predicha por el modelo binario

Despues de entrenar la red binaria, la usamos para estimar una caja de recorte (`ROI`) en las muestras multiclase.

Esta tabla de ROI es un puente entre la etapa 1 y la etapa 2:

- en `train` multiclase seguiremos usando ROI real para estabilidad
- en `val/test` multiclase usaremos ROI predicha para evaluar la cascada completa

In [ ]:
def predict_binary_rois_for_table(
    model: nn.Module,
    table: pd.DataFrame,
    input_size: tuple[int, int],
    threshold: float = 0.50,
) -> pd.DataFrame:
    records = []
    model.eval()
    with torch.no_grad():
        for _, row in table.reset_index(drop=True).iterrows():
            image_raw = read_gray(row['radiograph_path_abs'])
            original_shape = image_raw.shape
            image_resized = resize_image(image_raw, input_size).astype(np.float32) / 255.0
            image_resized = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)

            logits = model(image_resized)
            pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= threshold).astype(np.uint8)
            bbox_small = bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)

            if bbox_small is None:
                roi_meta = build_roi_record_from_binary_mask(
                    np.zeros(original_shape, dtype=np.uint8),
                    image_shape=original_shape,
                    roi_source='pred_binary_empty',
                )
            else:
                bbox_original = scale_bbox(bbox_small, src_shape=input_size, dst_shape=original_shape)
                bbox_original = expand_bbox(bbox_original, image_shape=original_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
                x0, y0, x1, y1 = bbox_original
                roi_meta = {
                    'bbox_x0': x0,
                    'bbox_y0': y0,
                    'bbox_x1': x1,
                    'bbox_y1': y1,
                    'bbox_width': x1 - x0,
                    'bbox_height': y1 - y0,
                    'roi_source': 'pred_binary',
                }

            records.append({'unique_sample_id': row['unique_sample_id'], **roi_meta})

    return pd.DataFrame(records)


multiclass_roi_df = predict_binary_rois_for_table(
    model=binary_model,
    table=multiclass_splits_df,
    input_size=IMG_SIZE_BINARY,
    threshold=BINARY_THRESHOLD,
)
multiclass_roi_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_binary_rois.csv'
multiclass_roi_df.to_csv(multiclass_roi_path, index=False)
roi_lookup = multiclass_roi_df.set_index('unique_sample_id').to_dict(orient='index')

display(multiclass_roi_df.head())
display(multiclass_roi_df['roi_source'].value_counts().rename_axis('roi_source').reset_index(name='images'))
print('Tabla de ROI guardada en:', multiclass_roi_path)

## Seccion 6. Entrenamiento multiclase sobre ROI espinal

Aqui aparece la mejora principal respecto al notebook anterior.

Antes:

- la red multiclase veia la radiografia completa

Ahora:

- la red multiclase ve una region centrada en la columna

Beneficios esperados:

- menos distraccion por fondo
- mejor uso de la resolucion efectiva
- una tarea mas cercana a "identificar vertebras" que a "buscar columna y clasificar al mismo tiempo"

Tambien usamos:

- `CrossEntropy` ponderada por frecuencia de clase
- `Dice loss` multiclase
- seleccion del mejor checkpoint por `val_macro_dice_fg`

In [ ]:
def estimate_multiclass_class_weights(table: pd.DataFrame) -> tuple[torch.Tensor, pd.DataFrame]:
    counts = np.ones(num_classes, dtype=np.float64)
    for _, row in table.reset_index(drop=True).iterrows():
        sample = prepare_multiclass_cascade_sample(
            row=row,
            output_size=IMG_SIZE_MULTICLASS,
            roi_mode='gt_binary',
            roi_lookup=None,
            apply_jitter=False,
        )
        mask = sample['mask']
        valid = mask != IGNORE_INDEX
        bincount = np.bincount(mask[valid].ravel(), minlength=num_classes)
        counts += bincount

    weights = counts.sum() / counts
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.25, 8.0)
    weights_df = pd.DataFrame(
        {
            'class_id': np.arange(num_classes),
            'class_name': class_names,
            'pixel_count': counts,
            'weight': weights,
        }
    )
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE), weights_df


multiclass_train_ds = CascadedThoracolumbarDataset(
    multiclass_splits_df.query("partition == 'train'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='gt_binary',
    roi_lookup=None,
    apply_jitter=True,
    apply_roi_augment=True,
)
multiclass_val_ds = CascadedThoracolumbarDataset(
    multiclass_splits_df.query("partition == 'val'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='pred_binary',
    roi_lookup=roi_lookup,
    apply_jitter=False,
)
multiclass_test_ds = CascadedThoracolumbarDataset(
    multiclass_splits_df.query("partition == 'test'"),
    image_size=IMG_SIZE_MULTICLASS,
    roi_mode='pred_binary',
    roi_lookup=roi_lookup,
    apply_jitter=False,
)

multiclass_train_loader = DataLoader(multiclass_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
multiclass_val_loader = DataLoader(multiclass_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
multiclass_test_loader = DataLoader(multiclass_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

class_weights, class_weights_df = estimate_multiclass_class_weights(
    multiclass_splits_df.query("partition == 'train'")
)
multiclass_loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)
multiclass_model = UNetSmall(in_channels=1, out_channels=num_classes).to(DEVICE)
# --- [Fase 3 adoptada] LR multiclase ×0,5 + CosineAnnealingLR (solo etapa multiclase) ---
_MULTICLASS_LR_SCALE = 0.5
LR_MULTICLASS = LR * _MULTICLASS_LR_SCALE
multiclass_optimizer = torch.optim.AdamW(
    multiclass_model.parameters(), lr=LR_MULTICLASS, weight_decay=WEIGHT_DECAY
)
multiclass_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    multiclass_optimizer, T_max=MULTICLASS_EPOCHS, eta_min=max(float(1e-8), LR_MULTICLASS * 0.01)
)

multiclass_history = []
best_multiclass_state = None
best_multiclass_val_macro_dice = -1.0
multiclass_start = time.time()

for epoch in range(1, MULTICLASS_EPOCHS + 1):
    multiclass_model.train()
    for batch in multiclass_train_loader:
        images = batch['image'].to(DEVICE)
        targets = batch['mask'].to(DEVICE)

        multiclass_optimizer.zero_grad()
        logits = multiclass_model(images)
        loss = multiclass_loss_fn(logits, targets) + dice_loss_multiclass(
            logits, targets, num_classes=num_classes, ignore_index=IGNORE_INDEX
        )
        loss.backward()
        multiclass_optimizer.step()

    train_metrics, _ = evaluate_multiclass(multiclass_model, multiclass_train_loader, loss_fn=multiclass_loss_fn)
    val_metrics, _ = evaluate_multiclass(multiclass_model, multiclass_val_loader, loss_fn=multiclass_loss_fn)
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_pixel_accuracy': train_metrics['pixel_accuracy'],
        'train_macro_dice_fg': train_metrics['macro_dice_fg'],
        'train_macro_iou_fg': train_metrics['macro_iou_fg'],
        'val_loss': val_metrics['loss'],
        'val_pixel_accuracy': val_metrics['pixel_accuracy'],
        'val_macro_dice_fg': val_metrics['macro_dice_fg'],
        'val_macro_iou_fg': val_metrics['macro_iou_fg'],
    }
    multiclass_history.append(row)

    if val_metrics['macro_dice_fg'] > best_multiclass_val_macro_dice:
        best_multiclass_val_macro_dice = val_metrics['macro_dice_fg']
        best_multiclass_state = {k: v.detach().cpu().clone() for k, v in multiclass_model.state_dict().items()}

    print(
        f"[Cascade-Multiclass][Epoch {epoch:02d}/{MULTICLASS_EPOCHS}] "
        f"train_macro_dice={row['train_macro_dice_fg']:.4f} "
        f"val_macro_dice={row['val_macro_dice_fg']:.4f} "
        f"train_macro_iou={row['train_macro_iou_fg']:.4f} "
        f"val_macro_iou={row['val_macro_iou_fg']:.4f}"
    )
    multiclass_scheduler.step()

multiclass_history_df = pd.DataFrame(multiclass_history)
multiclass_model.load_state_dict(best_multiclass_state)
multiclass_test_metrics, multiclass_per_class_df = evaluate_multiclass(
    multiclass_model,
    multiclass_test_loader,
    loss_fn=multiclass_loss_fn,
)
multiclass_elapsed_min = (time.time() - multiclass_start) / 60.0

multiclass_model_path = MODEL_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_cascade_fase7_lastvis_cuda_best.pt'
multiclass_history_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_history.csv'
multiclass_test_metrics_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_test_metrics.csv'
multiclass_per_class_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_per_class_metrics.csv'
multiclass_class_weights_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_class_weights.csv'

torch.save(multiclass_model.state_dict(), multiclass_model_path)
multiclass_history_df.to_csv(multiclass_history_path, index=False)
pd.DataFrame([multiclass_test_metrics]).to_csv(multiclass_test_metrics_path, index=False)
multiclass_per_class_df.to_csv(multiclass_per_class_path, index=False)
class_weights_df.to_csv(multiclass_class_weights_path, index=False)

print('\nMejor val_macro_dice_fg:', round(best_multiclass_val_macro_dice, 4))
print('Metricas finales en test:', multiclass_test_metrics)
print('Tiempo de entrenamiento multiclase (min):', round(multiclass_elapsed_min, 2))

display(class_weights_df)
display(multiclass_history_df)
display(pd.DataFrame([multiclass_test_metrics]))
display(multiclass_per_class_df.sort_values('dice', ascending=False))
plot_history(multiclass_history_df, f'Historia de entrenamiento: cascada thoracolumbar {MULTICLASS_SUBSET}')

## Seccion 7. Visualizacion cualitativa

Las metricas son necesarias, pero no suficientes.

En problemas de imagen medica siempre conviene revisar ejemplos visuales para responder preguntas como:

- la binaria recorta bien la columna?
- la ROI realmente enfoca la zona util?
- la multiclase esta confundiendo vertebras vecinas?
- el modelo sobre-segmenta o sub-segmenta?

In [ ]:
def show_binary_predictions(model: nn.Module, dataset: Dataset, n: int = 3) -> None:
    model.eval()
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for idx in range(n):
            sample = dataset[idx]
            image = sample['image'].unsqueeze(0).to(DEVICE)
            target = sample['mask'][0].numpy()
            pred = (torch.sigmoid(model(image))[0, 0].detach().cpu().numpy() >= BINARY_THRESHOLD).astype(np.uint8)
            axes[idx, 0].imshow(sample['image'][0].numpy(), cmap='gray')
            axes[idx, 0].set_title(f"Imagen: {sample['sample_id']}")
            axes[idx, 1].imshow(target, cmap='gray')
            axes[idx, 1].set_title('GT binaria')
            axes[idx, 2].imshow(pred, cmap='gray')
            axes[idx, 2].set_title('Pred binaria')
            for j in range(3):
                axes[idx, j].axis('off')
    plt.tight_layout()
    plt.show()


def show_cascade_multiclass_predictions(model: nn.Module, dataset: Dataset, n: int = 3) -> None:
    model.eval()
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for idx in range(n):
            sample = dataset[idx]
            image = sample['image'].unsqueeze(0).to(DEVICE)
            pred = torch.argmax(model(image), dim=1)[0].detach().cpu().numpy()
            target = sample['mask'].numpy().copy()
            target[target == IGNORE_INDEX] = 0
            axes[idx, 0].imshow(sample['image'][0].numpy(), cmap='gray')
            axes[idx, 0].set_title(f"ROI: {sample['sample_id']}")
            axes[idx, 1].imshow(target, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
            axes[idx, 1].set_title('GT multiclase')
            axes[idx, 2].imshow(pred, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
            axes[idx, 2].set_title('Pred multiclase')
            for j in range(3):
                axes[idx, j].axis('off')
    plt.tight_layout()
    plt.show()


show_binary_predictions(binary_model, binary_test_ds, n=3)
show_cascade_multiclass_predictions(multiclass_model, multiclass_test_ds, n=3)

### [FASE 7] Reanudar tras parches del notebook

Si ya corriste cascada + last_visible: (1) celda **config**, (2) desde la celda que falló. No repitas entrenamiento salvo reinicio de kernel.


In [ ]:
TARGET_SUBSET = MULTICLASS_SUBSET  # alias notebook 07


### [FASE 7] Reanudar tras parches del notebook

Si **ya** corriste cascada + last_visible y solo actualizaste el `.ipynb`:

1. Ejecuta la celda **config** (imports + aliases).
2. Continúa desde la celda que falló (clipping / métricas / export).

**No** hace falta repetir entrenamiento salvo **Runtime → Reiniciar** el kernel.


In [ ]:
# --- [FASE 7] Aliases idempotentes (ejecutar si solo re-corres §8+) ---
TARGET_SUBSET = MULTICLASS_SUBSET

## Seccion 8. Fase 7 — Metadata y target `last_visible_idx`

Target supervisado a partir de mascaras GT; mismo split que `multiclass_splits_df`.


In [ ]:
# --- [FASE 7] Target last_visible_idx (mismo split que multiclass_splits_df) ---
canonical_labels = TARGET_LABELS
label_to_class_id = {label: class_names.index(label) for label in canonical_labels}


def extract_visible_range_from_mask(path: str) -> tuple[int | None, int | None, list[str]]:
    raw = np.array(Image.open(path), dtype=np.int32)
    labels_present: list[str] = []
    for rid in sorted(int(x) for x in np.unique(raw) if int(x) > 0):
        label = multiclass_map.get(rid)
        if label in canonical_labels:
            labels_present.append(label)
    if not labels_present:
        return None, None, []
    first_idx = canonical_labels.index(labels_present[0])
    last_idx = canonical_labels.index(labels_present[-1])
    return first_idx, last_idx, labels_present


work_df = multiclass_splits_df.copy()
first_last = work_df['multiclass_mask_path_abs'].apply(extract_visible_range_from_mask)
work_df['first_visible_idx'] = [item[0] for item in first_last]
work_df['last_visible_idx'] = [item[1] for item in first_last]
work_df['visible_labels_gt'] = [', '.join(item[2]) for item in first_last]
work_df = work_df.loc[
    work_df['first_visible_idx'].notna() & work_df['last_visible_idx'].notna()
].copy().reset_index(drop=True)
work_df['first_visible_idx'] = work_df['first_visible_idx'].astype(int)
work_df['last_visible_idx'] = work_df['last_visible_idx'].astype(int)
work_df['first_visible_label'] = work_df['first_visible_idx'].map(lambda idx: canonical_labels[int(idx)])
work_df['last_visible_label'] = work_df['last_visible_idx'].map(lambda idx: canonical_labels[int(idx)])

print('Muestras para last-visible estimator:', len(work_df))
display(work_df.groupby('partition').size().rename('images').reset_index())
display(work_df[['partition', 'split', 'image', 'first_visible_label', 'last_visible_label']].head(10))

### [FASE 7] Distribucion del target `last_visible_idx`


Este paso es importante porque aqui realmente esta la dificultad del problema.

In [ ]:
last_dist_df = (
    work_df.groupby(['partition', 'last_visible_label'])
    .size()
    .rename('images')
    .reset_index()
    .sort_values(['partition', 'images'], ascending=[True, False])
)
display(last_dist_df.head(30))

### [FASE 7] Utilidades ROI / features auxiliares

Reutiliza helpers del baseline; anade normalizacion y canales de coordenadas.


In [ ]:
# --- [FASE 7] Helpers adicionales (ROI / last_visible) ---


def normalize_image(image_2d: np.ndarray) -> np.ndarray:
    mean = float(image_2d.mean())
    std = float(image_2d.std())
    if std < 1e-6:
        return image_2d - mean
    return (image_2d - mean) / std


def build_coordinate_channels(height: int, width: int) -> np.ndarray:
    y_coords = np.linspace(0.0, 1.0, height, dtype=np.float32)[:, None]
    x_coords = np.linspace(0.0, 1.0, width, dtype=np.float32)[None, :]
    y_map = np.repeat(y_coords, width, axis=1)
    x_map = np.repeat(x_coords, height, axis=0)
    return np.stack([y_map, x_map], axis=0)

### [FASE 7] ROI e inferencia multiclase (cascada de este notebook)

Usa `binary_model` y `multiclass_model` entrenados arriba (no checkpoints de otras fases).


In [ ]:
# --- [FASE 7] Inferencia cascada (modelos de secciones 4-6) y tabla prep_df ---
binary_model.eval()
multiclass_model.eval()


def predict_binary_bbox_from_image(image_raw: np.ndarray) -> tuple[int, int, int, int] | None:
    image_resized = resize_image(image_raw, IMG_SIZE_BINARY).astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = binary_model(image_tensor)
        pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= BINARY_THRESHOLD).astype(np.uint8)
    return bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)


def infer_multiclass_on_bbox(image_raw: np.ndarray, bbox: tuple[int, int, int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    image_crop = crop_array(image_raw, bbox)
    image_crop = resize_image(image_crop, IMG_SIZE_MULTICLASS).astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_crop[None, None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = multiclass_model(image_tensor)
        probs = torch.softmax(logits, dim=1)
        if USE_MULTICLASS_TTA:
            flipped_input = torch.flip(image_tensor, dims=[3])
            flipped_logits = multiclass_model(flipped_input)
            flipped_probs = torch.softmax(flipped_logits, dim=1)
            flipped_probs = torch.flip(flipped_probs, dims=[3])
            probs = 0.5 * (probs + flipped_probs)
        probs_np = probs[0].detach().cpu().numpy().astype(np.float32)
        pred_mask = np.argmax(probs_np, axis=0).astype(np.int64)
    return image_crop, pred_mask, probs_np


def extract_aux_features_from_prediction(pred_mask: np.ndarray, prob_map: np.ndarray) -> np.ndarray:
    h, w = pred_mask.shape
    fg_mask = (pred_mask > 0).astype(np.float32)
    total_fg = float(fg_mask.sum()) + 1e-6

    presence = []
    area_ratio = []
    centroid_y = []
    y_min_norm = []
    y_max_norm = []
    height_span_norm = []
    mean_confidence = []
    for label in canonical_labels:
        class_id = class_names.index(label)
        class_mask = pred_mask == class_id
        area = float(class_mask.sum())
        presence.append(1.0 if area >= PRESENCE_THRESHOLD_PIXELS else 0.0)
        area_ratio.append(area / total_fg)
        if area > 0:
            ys, _ = np.where(class_mask)
            centroid_y.append(float(np.mean(ys) / max(h - 1, 1)))
            y_min_norm.append(float(np.min(ys) / max(h - 1, 1)))
            y_max_norm.append(float(np.max(ys) / max(h - 1, 1)))
            height_span_norm.append(float((np.max(ys) - np.min(ys) + 1) / max(h, 1)))
            mean_confidence.append(float(prob_map[class_id][class_mask].mean()))
        else:
            centroid_y.append(0.0)
            y_min_norm.append(0.0)
            y_max_norm.append(0.0)
            height_span_norm.append(0.0)
            mean_confidence.append(0.0)

    pred_present_indices = [i for i, p in enumerate(presence) if p > 0.5]
    min_present_idx = float(min(pred_present_indices)) if pred_present_indices else 0.0
    max_present_idx = float(max(pred_present_indices)) if pred_present_indices else 0.0
    num_present = float(len(pred_present_indices))

    row_profile = fg_mask.sum(axis=1).astype(np.float32)
    if row_profile.max() > 0:
        row_profile = row_profile / row_profile.max()
    binned_profile = np.array_split(row_profile, PROFILE_BINS)
    profile_features = [float(chunk.mean()) for chunk in binned_profile]

    return np.array(
        presence
        + area_ratio
        + centroid_y
        + y_min_norm
        + y_max_norm
        + height_span_norm
        + mean_confidence
        + [
            min_present_idx / (len(canonical_labels) - 1),
            max_present_idx / (len(canonical_labels) - 1),
            num_present / len(canonical_labels),
        ]
        + profile_features,
        dtype=np.float32,
    )


def estimate_last_visible_from_mask(pred_mask: np.ndarray) -> int:
    present_indices = [
        canonical_labels.index(class_names[int(class_id)])
        for class_id in sorted(int(x) for x in np.unique(pred_mask) if int(x) > 0)
        if int(class_id) < len(class_names) and class_names[int(class_id)] in canonical_labels
    ]
    if not present_indices:
        return 0
    return int(max(present_indices))


prep_rows = []
image_crop_lookup = {}
raw_pred_lookup_all = {}
target_lookup_all = {}

for _, row in work_df.iterrows():
    image_raw = read_gray(row['radiograph_path_abs'])
    image_shape = image_raw.shape

    if row['partition'] == 'train':
        gt_binary = build_binary_mask(row['binary_mask_path_abs'], size=None)
        bbox = bbox_from_mask(gt_binary, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
        roi_source = 'gt_binary'
    else:
        bbox_small = predict_binary_bbox_from_image(image_raw)
        bbox = scale_bbox(bbox_small, src_shape=IMG_SIZE_BINARY, dst_shape=image_shape) if bbox_small is not None else None
        roi_source = 'pred_binary'

    if bbox is None:
        x0, y0, x1, y1 = 0, 0, image_shape[1], image_shape[0]
        roi_source = f'{roi_source}_fallback_full_image'
    else:
        x0, y0, x1, y1 = expand_bbox(bbox, image_shape=image_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)

    bbox_final = (int(x0), int(y0), int(x1), int(y1))
    image_crop_2d, raw_pred_mask, raw_prob_map = infer_multiclass_on_bbox(image_raw, bbox_final)
    aux_features = extract_aux_features_from_prediction(raw_pred_mask, raw_prob_map)
    heuristic_last_idx = estimate_last_visible_from_mask(raw_pred_mask)

    target_full = build_multiclass_mask(row['multiclass_mask_path_abs'], size=None)
    target_crop = crop_array(target_full, bbox_final)
    target_crop = resize_mask(target_crop, IMG_SIZE_MULTICLASS).astype(np.int64)

    prep_rows.append({
        'unique_sample_id': row['unique_sample_id'],
        'partition': row['partition'],
        'split': row['split'],
        'image': row['image'],
        'radiograph_path_abs': row['radiograph_path_abs'],
        'multiclass_mask_path_abs': row['multiclass_mask_path_abs'],
        'first_visible_idx': int(row['first_visible_idx']),
        'last_visible_idx': int(row['last_visible_idx']),
        'first_visible_label': row['first_visible_label'],
        'last_visible_label': row['last_visible_label'],
        'heuristic_last_idx': heuristic_last_idx,
        'bbox_x0': bbox_final[0],
        'bbox_y0': bbox_final[1],
        'bbox_x1': bbox_final[2],
        'bbox_y1': bbox_final[3],
        'roi_source': roi_source,
        'aux_features': aux_features,
    })

    image_crop_lookup[row['unique_sample_id']] = image_crop_2d
    raw_pred_lookup_all[row['unique_sample_id']] = raw_pred_mask
    target_lookup_all[row['unique_sample_id']] = target_crop

prep_df = pd.DataFrame(prep_rows)
aux_feature_matrix = np.stack([row['aux_features'] for row in prep_rows], axis=0)
aux_feature_dim = int(aux_feature_matrix.shape[1])

print('prep_df:', prep_df.shape)
print('aux_feature_dim:', aux_feature_dim)
display(prep_df.groupby(['partition', 'roi_source']).size().rename('images').reset_index())

### [FASE 7] 6. Dataset fusionado y arquitectura del estimador de ultima vertebra visible

El modelo combina:

- encoder CNN sobre la ROI
- MLP sobre features anatomicas auxiliares
- fusion final para clasificar `last_visible_idx`

In [ ]:
partition_to_indices = {
    part: prep_df.index[prep_df["partition"] == part].to_numpy()
    for part in ["train", "val", "test"]
}

train_aux_mean = aux_feature_matrix[partition_to_indices["train"]].mean(axis=0)
train_aux_std = aux_feature_matrix[partition_to_indices["train"]].std(axis=0)
train_aux_std = np.where(train_aux_std < 1e-6, 1.0, train_aux_std)


class LastVisibleDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, aux_features: np.ndarray, aux_mean: np.ndarray, aux_std: np.ndarray):
        self.frame = frame.reset_index(drop=True).copy()
        self.aux_features = aux_features
        self.aux_mean = aux_mean.astype(np.float32)
        self.aux_std = aux_std.astype(np.float32)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        sample_id = row["unique_sample_id"]
        image_crop = image_crop_lookup[sample_id]
        image_small = resize_image((image_crop * 255.0).astype(np.uint8), IMG_SIZE_LAST).astype(np.float32) / 255.0
        image_small = normalize_image(image_small)
        coords = build_coordinate_channels(IMG_SIZE_LAST[0], IMG_SIZE_LAST[1])
        image_tensor = np.concatenate([np.expand_dims(image_small, axis=0), coords], axis=0)

        aux = self.aux_features[int(row["row_idx"])]
        aux = ((aux - self.aux_mean) / self.aux_std).astype(np.float32)
        return {
            "image": torch.tensor(image_tensor, dtype=torch.float32),
            "aux": torch.tensor(aux, dtype=torch.float32),
            "last_idx": torch.tensor(int(row["last_visible_idx"]), dtype=torch.long),
            "first_idx": torch.tensor(int(row["first_visible_idx"]), dtype=torch.long),
            "heuristic_last_idx": torch.tensor(int(row["heuristic_last_idx"]), dtype=torch.long),
            "sample_id": sample_id,
        }


class ConvBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LastVisibleEstimator(nn.Module):
    def __init__(self, aux_dim: int, num_labels: int = 17, dropout: float = 0.25):
        super().__init__()
        self.image_encoder = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.image_pool = nn.AdaptiveAvgPool2d(1)
        self.image_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.aux_head = nn.Sequential(
            nn.Linear(aux_dim, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )
        self.fusion = nn.Sequential(
            nn.Linear(192 + 96, 160),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(160, num_labels),
        )

    def forward(self, image: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        image_feat = self.image_encoder(image)
        image_feat = self.image_pool(image_feat)
        image_feat = self.image_head(image_feat)
        aux_feat = self.aux_head(aux)
        fused = torch.cat([image_feat, aux_feat], dim=1)
        return self.fusion(fused)


prep_df = prep_df.reset_index(drop=True).copy()
prep_df["row_idx"] = np.arange(len(prep_df))

train_frame = prep_df.loc[prep_df["partition"] == "train"].copy().reset_index(drop=True)
val_frame = prep_df.loc[prep_df["partition"] == "val"].copy().reset_index(drop=True)
test_frame = prep_df.loc[prep_df["partition"] == "test"].copy().reset_index(drop=True)

train_dataset = LastVisibleDataset(train_frame, aux_feature_matrix, train_aux_mean, train_aux_std)
val_dataset = LastVisibleDataset(val_frame, aux_feature_matrix, train_aux_mean, train_aux_std)
test_dataset = LastVisibleDataset(test_frame, aux_feature_matrix, train_aux_mean, train_aux_std)

train_loader = DataLoader(train_dataset, batch_size=LAST_BATCH_SIZE, shuffle=True, num_workers=LAST_NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=LAST_BATCH_SIZE, shuffle=False, num_workers=LAST_NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=LAST_BATCH_SIZE, shuffle=False, num_workers=LAST_NUM_WORKERS, pin_memory=PIN_MEMORY)

last_class_counts = np.ones(len(canonical_labels), dtype=np.float32)
last_class_counts += np.bincount(train_frame["last_visible_idx"].to_numpy(), minlength=len(canonical_labels)).astype(np.float32)
last_class_weights = np.power(last_class_counts.sum() / last_class_counts, 0.5)
last_class_weights = last_class_weights / last_class_weights.mean()
last_class_weights = np.clip(last_class_weights, 0.50, 3.50)
last_class_weights = torch.tensor(last_class_weights, dtype=torch.float32, device=DEVICE)

print("train:", len(train_dataset), "val:", len(val_dataset), "test:", len(test_dataset))


### [FASE 7] 7. Entrenamiento del estimador especializado

El criterio principal de seleccion del mejor modelo sera:

- `val_within1_acc`

porque en este problema un error de una sola vertebra sigue siendo muy util para
clipping anatomico.

In [ ]:
def expectation_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    class_axis = torch.arange(logits.shape[1], dtype=torch.float32, device=logits.device)
    expected = (torch.softmax(logits, dim=1) * class_axis.unsqueeze(0)).sum(dim=1)
    return torch.mean(torch.abs(expected - targets.float()))


def blend_last_prediction(
    logits: torch.Tensor,
    first_idx: torch.Tensor,
    heuristic_last_idx: torch.Tensor,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    probs = torch.softmax(logits, dim=1)
    argmax_idx = torch.argmax(probs, dim=1)
    class_axis = torch.arange(probs.shape[1], dtype=torch.float32, device=probs.device)
    expected_idx = (probs * class_axis.unsqueeze(0)).sum(dim=1)
    blended = (
        (1.0 - LAST_EXPECTATION_BLEND - LAST_HEURISTIC_BLEND) * argmax_idx.float()
        + LAST_EXPECTATION_BLEND * expected_idx
        + LAST_HEURISTIC_BLEND * heuristic_last_idx.float()
    )
    blended = torch.round(blended).long()
    blended = torch.maximum(blended, first_idx.long())
    blended = torch.clamp(blended, 0, len(canonical_labels) - 1)
    return (
        blended.detach().cpu().numpy(),
        argmax_idx.detach().cpu().numpy(),
        expected_idx.detach().cpu().numpy(),
    )


def evaluate_last_model(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[dict, pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_samples = 0
    rows = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
            aux = batch["aux"].to(DEVICE, non_blocking=PIN_MEMORY)
            targets = batch["last_idx"].to(DEVICE, non_blocking=PIN_MEMORY)
            first_idx = batch["first_idx"].to(DEVICE, non_blocking=PIN_MEMORY)
            heuristic_last_idx = batch["heuristic_last_idx"].to(DEVICE, non_blocking=PIN_MEMORY)

            logits = model(images, aux)
            loss = loss_fn(logits, targets) + LAST_EXPECTATION_LOSS_WEIGHT * expectation_loss(logits, targets)

            batch_size = images.size(0)
            total_loss += float(loss.item()) * batch_size
            total_samples += batch_size

            preds, argmax_idx, expected_idx = blend_last_prediction(logits, first_idx, heuristic_last_idx)
            truth = targets.detach().cpu().numpy()
            first_idx_np = first_idx.detach().cpu().numpy()
            heuristic_np = heuristic_last_idx.detach().cpu().numpy()
            for sample_id, pred_idx, raw_argmax, raw_expected, true_idx, first_visible_idx, heuristic_idx in zip(
                batch["sample_id"],
                preds,
                argmax_idx,
                expected_idx,
                truth,
                first_idx_np,
                heuristic_np,
            ):
                rows.append({
                    "unique_sample_id": sample_id,
                    "first_idx": int(first_visible_idx),
                    "heuristic_last_idx": int(heuristic_idx),
                    "last_true_idx": int(true_idx),
                    "last_pred_idx": int(pred_idx),
                    "last_argmax_idx": int(raw_argmax),
                    "last_expected_idx": float(raw_expected),
                    "first_label": canonical_labels[int(first_visible_idx)],
                    "last_true_label": canonical_labels[int(true_idx)],
                    "last_pred_label": canonical_labels[int(pred_idx)],
                })

    pred_df = pd.DataFrame(rows)
    pred_df["last_abs_error"] = (pred_df["last_pred_idx"] - pred_df["last_true_idx"]).abs()
    pred_df["last_exact_match"] = pred_df["last_pred_idx"] == pred_df["last_true_idx"]
    pred_df["last_within1_match"] = pred_df["last_abs_error"] <= 1
    pred_df["last_overprediction"] = pred_df["last_pred_idx"] > pred_df["last_true_idx"]
    pred_df["last_underprediction"] = pred_df["last_pred_idx"] < pred_df["last_true_idx"]

    metrics = {
        "loss": float(total_loss / max(total_samples, 1)),
        "last_acc": float(pred_df["last_exact_match"].mean()),
        "last_within1_acc": float(pred_df["last_within1_match"].mean()),
        "last_mae": float(pred_df["last_abs_error"].mean()),
        "overprediction_rate": float(pred_df["last_overprediction"].mean()),
        "underprediction_rate": float(pred_df["last_underprediction"].mean()),
    }
    return metrics, pred_df


model = LastVisibleEstimator(aux_dim=aux_feature_dim, num_labels=len(canonical_labels), dropout=LAST_DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LAST_LR, weight_decay=LAST_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
loss_fn = nn.CrossEntropyLoss(weight=last_class_weights, label_smoothing=LAST_LABEL_SMOOTHING)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_state = None
best_score = -1.0
patience_counter = 0
history_rows = []
best_model_path = MODEL_DIR / 'last_visible_estimator_fase7_lastvis_cuda_best.pt'

for epoch in range(1, LAST_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    epoch_start = time.time()

    for batch in train_loader:
        images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
        aux = batch["aux"].to(DEVICE, non_blocking=PIN_MEMORY)
        targets = batch["last_idx"].to(DEVICE, non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images, aux)
            loss = loss_fn(logits, targets) + LAST_EXPECTATION_LOSS_WEIGHT * expectation_loss(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), LAST_GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        train_loss_sum += float(loss.item()) * batch_size
        train_count += batch_size

    train_loss = float(train_loss_sum / max(train_count, 1))
    val_metrics, _ = evaluate_last_model(model, val_loader, loss_fn)
    scheduler.step(val_metrics["last_within1_acc"])

    history_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_last_acc": val_metrics["last_acc"],
        "val_last_within1_acc": val_metrics["last_within1_acc"],
        "val_last_mae": val_metrics["last_mae"],
        "val_overprediction_rate": val_metrics["overprediction_rate"],
        "val_underprediction_rate": val_metrics["underprediction_rate"],
        "lr": optimizer.param_groups[0]["lr"],
        "epoch_seconds": time.time() - epoch_start,
    }
    history_rows.append(history_row)
    print(history_row)

    score = val_metrics["last_within1_acc"]
    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, best_model_path)
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= LAST_PATIENCE:
        print("Early stopping activado.")
        break

if best_state is None:
    raise RuntimeError("No se pudo entrenar el last-visible estimator.")

model.load_state_dict(best_state)
history_df = pd.DataFrame(history_rows)
display(history_df.tail())
print("Best model saved to:", best_model_path)


### [FASE 7] 8. Evaluacion del estimador `last_visible_idx`

Nos interesan especialmente:

- accuracy exacta
- accuracy within `+-1`
- MAE
- tasa de sobreprediccion

In [ ]:
train_metrics, train_pred_df = evaluate_last_model(model, train_loader, loss_fn)
val_metrics, val_pred_df = evaluate_last_model(model, val_loader, loss_fn)
test_metrics, test_pred_df = evaluate_last_model(model, test_loader, loss_fn)

last_summary_df = pd.DataFrame([
    {'partition': 'train', **train_metrics},
    {'partition': 'val', **val_metrics},
    {'partition': 'test', **test_metrics},
])

display(last_summary_df)
display(test_pred_df.head(15))

### [FASE 7] 9. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['val_last_acc'], label='val_exact')
axes[1].plot(history_df['epoch'], history_df['val_last_within1_acc'], label='val_within1')
axes[1].set_title('Last Visible Accuracy')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['val_overprediction_rate'], label='overpred')
axes[2].plot(history_df['epoch'], history_df['val_underprediction_rate'], label='underpred')
axes[2].set_title('Prediction Bias')
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### [FASE 7] 10. Clipping anatomico usando la ultima vertebra visible predicha

Comparamos:

- `raw`
- `prev_range_clip` de la etapa anterior, si el archivo existe
- `last_pred_clip`
- `oracle_clip`

In [ ]:
prev_range_lookup = {}

if PREV_RANGE_TEST_PATH.exists():
    prev_range_df = pd.read_csv(PREV_RANGE_TEST_PATH)
    prev_range_lookup = prev_range_df.set_index("unique_sample_id").to_dict(orient="index")
    print("Se cargo referencia de visible-range estimator anterior.")
else:
    print("No se encontro archivo de range estimator anterior. Se omitira esa comparacion.")


def clip_mask_to_last_idx(mask_2d: np.ndarray, first_idx: int, last_idx: int) -> np.ndarray:
    first_idx = int(first_idx)
    last_idx = int(max(last_idx, first_idx))
    allowed_labels = canonical_labels[first_idx:last_idx + 1]
    allowed_ids = {label_to_class_id[label] for label in allowed_labels}
    out = np.zeros_like(mask_2d, dtype=np.int64)
    for class_id in allowed_ids:
        out[mask_2d == class_id] = class_id
    return out


test_gt_lookup = test_frame.set_index("unique_sample_id")[["first_visible_idx", "last_visible_idx"]].to_dict(orient="index")
last_pred_lookup = test_pred_df.set_index("unique_sample_id").to_dict(orient="index")

raw_pred_lookup = {}
oracle_clip_lookup = {}
last_pred_clip_lookup = {}
prev_range_clip_lookup = {}
target_lookup = {}
image_lookup = {}

for _, row in test_frame.iterrows():
    sample_id = row["unique_sample_id"]
    raw_mask = raw_pred_lookup_all[sample_id]
    target_mask = target_lookup_all[sample_id]
    image_roi = image_crop_lookup[sample_id]

    gt_first = int(test_gt_lookup[sample_id]["first_visible_idx"])
    gt_last = int(test_gt_lookup[sample_id]["last_visible_idx"])
    pred_last = int(last_pred_lookup[sample_id]["last_pred_idx"])

    raw_pred_lookup[sample_id] = raw_mask
    oracle_clip_lookup[sample_id] = clip_mask_to_last_idx(raw_mask, gt_first, gt_last)
    last_pred_clip_lookup[sample_id] = clip_mask_to_last_idx(raw_mask, gt_first, pred_last)
    target_lookup[sample_id] = target_mask
    image_lookup[sample_id] = image_roi

    if sample_id in prev_range_lookup:
        prev_first = int(prev_range_lookup[sample_id]["first_pred_idx"])
        prev_last = int(prev_range_lookup[sample_id]["last_pred_idx"])
        prev_range_clip_lookup[sample_id] = clip_mask_to_last_idx(raw_mask, prev_first, prev_last)

print("Muestras de test preparadas para clipping:", len(raw_pred_lookup))


### [FASE 7] 11. Metricas de segmentacion comparadas

In [ ]:
def metrics_from_prediction_lookup(pred_lookup: dict[str, np.ndarray], target_lookup_in: dict[str, np.ndarray]) -> tuple[pd.DataFrame, pd.DataFrame]:
    intersection = np.zeros(num_classes, dtype=np.float64)
    pred_area = np.zeros(num_classes, dtype=np.float64)
    target_area = np.zeros(num_classes, dtype=np.float64)
    total_valid_correct = 0.0
    total_valid_pixels = 0.0

    for sample_id, pred_mask in pred_lookup.items():
        target_mask = target_lookup_in[sample_id]
        valid = target_mask != IGNORE_INDEX
        total_valid_correct += float((pred_mask[valid] == target_mask[valid]).sum())
        total_valid_pixels += float(valid.sum())
        for class_id in range(num_classes):
            pred_class = pred_mask[valid] == class_id
            target_class = target_mask[valid] == class_id
            intersection[class_id] += np.logical_and(pred_class, target_class).sum()
            pred_area[class_id] += pred_class.sum()
            target_area[class_id] += target_class.sum()

    dice = (2.0 * intersection + 1e-6) / (pred_area + target_area + 1e-6)
    iou = (intersection + 1e-6) / (pred_area + target_area - intersection + 1e-6)
    per_class_df = pd.DataFrame({
        'class_id': np.arange(num_classes),
        'class_name': class_names,
        'pred_pixels': pred_area,
        'target_pixels': target_area,
        'dice': dice,
        'iou': iou,
    })
    per_class_df['region'] = per_class_df['class_name'].map(
        lambda x: 'background' if x == 'background' else ('thoracic' if x.startswith('T') else 'lumbar')
    )
    fg_df = per_class_df.loc[per_class_df['class_id'] > 0].copy()
    summary_df = pd.DataFrame([
        {'metric': 'pixel_accuracy', 'value': float((total_valid_correct + 1e-6) / (total_valid_pixels + 1e-6))},
        {'metric': 'macro_dice_fg', 'value': float(fg_df['dice'].mean())},
        {'metric': 'macro_iou_fg', 'value': float(fg_df['iou'].mean())},
        {'metric': 'macro_dice_thoracic', 'value': float(fg_df.loc[fg_df['region'] == 'thoracic', 'dice'].mean())},
        {'metric': 'macro_dice_lumbar', 'value': float(fg_df.loc[fg_df['region'] == 'lumbar', 'dice'].mean())},
    ])
    return summary_df, per_class_df


raw_summary_df, raw_per_class_df = metrics_from_prediction_lookup(raw_pred_lookup, target_lookup)
oracle_summary_df, oracle_per_class_df = metrics_from_prediction_lookup(oracle_clip_lookup, target_lookup)
last_summary_seg_df, last_per_class_df = metrics_from_prediction_lookup(last_pred_clip_lookup, target_lookup)

summary_compare_df = (
    raw_summary_df.rename(columns={'value': 'raw'})
    .merge(oracle_summary_df.rename(columns={'value': 'oracle_clip'}), on='metric')
    .merge(last_summary_seg_df.rename(columns={'value': 'last_pred_clip'}), on='metric')
)
if prev_range_clip_lookup:
    prev_summary_df, prev_per_class_df = metrics_from_prediction_lookup(prev_range_clip_lookup, target_lookup)
    summary_compare_df = summary_compare_df.merge(prev_summary_df.rename(columns={'value': 'prev_range_clip'}), on='metric')
else:
    prev_per_class_df = pd.DataFrame()

display(summary_compare_df)

### [FASE 7] 12. Analisis por muestra: extra labels y missing labels

Esta tabla nos dira si el nuevo clipping reduce la sobreprediccion sin destruir
la segmentacion util.

In [ ]:
def present_labels_from_mask(mask_2d: np.ndarray) -> list[str]:
    ids = sorted([int(x) for x in np.unique(mask_2d) if int(x) > 0])
    return [class_names[i] for i in ids]


rows = []
for _, row in test_frame.iterrows():
    sample_id = row['unique_sample_id']
    target_mask = target_lookup[sample_id]
    raw_mask = raw_pred_lookup[sample_id]
    oracle_mask = oracle_clip_lookup[sample_id]
    last_mask = last_pred_clip_lookup[sample_id]

    gt_labels = present_labels_from_mask(np.where(target_mask == IGNORE_INDEX, 0, target_mask))
    raw_labels = present_labels_from_mask(raw_mask)
    oracle_labels = present_labels_from_mask(oracle_mask)
    last_labels = present_labels_from_mask(last_mask)

    record = {
        'unique_sample_id': sample_id,
        'split': row['split'],
        'image': row['image'],
        'gt_first_label': row['first_visible_label'],
        'gt_last_label': row['last_visible_label'],
        'last_pred_label': last_pred_lookup[sample_id]['last_pred_label'],
        'gt_labels': ', '.join(gt_labels),
        'raw_labels': ', '.join(raw_labels),
        'oracle_labels': ', '.join(oracle_labels),
        'last_pred_clip_labels': ', '.join(last_labels),
        'raw_extra_count': len(set(raw_labels) - set(gt_labels)),
        'oracle_extra_count': len(set(oracle_labels) - set(gt_labels)),
        'last_extra_count': len(set(last_labels) - set(gt_labels)),
        'raw_missing_count': len(set(gt_labels) - set(raw_labels)),
        'oracle_missing_count': len(set(oracle_labels) - set(gt_labels)),
        'last_missing_count': len(set(gt_labels) - set(last_labels)),
    }
    if sample_id in prev_range_clip_lookup:
        prev_labels = present_labels_from_mask(prev_range_clip_lookup[sample_id])
        record['prev_range_clip_labels'] = ', '.join(prev_labels)
        record['prev_extra_count'] = len(set(prev_labels) - set(gt_labels))
        record['prev_missing_count'] = len(set(gt_labels) - set(prev_labels))
    rows.append(record)

per_sample_compare_df = pd.DataFrame(rows)
per_sample_compare_df['last_extra_reduction_vs_raw'] = per_sample_compare_df['raw_extra_count'] - per_sample_compare_df['last_extra_count']
per_sample_compare_df['last_missing_change_vs_raw'] = per_sample_compare_df['last_missing_count'] - per_sample_compare_df['raw_missing_count']

summary_rows = [
    {'metric': 'mean_raw_extra_count', 'value': float(per_sample_compare_df['raw_extra_count'].mean())},
    {'metric': 'mean_oracle_extra_count', 'value': float(per_sample_compare_df['oracle_extra_count'].mean())},
    {'metric': 'mean_last_extra_count', 'value': float(per_sample_compare_df['last_extra_count'].mean())},
    {'metric': 'mean_raw_missing_count', 'value': float(per_sample_compare_df['raw_missing_count'].mean())},
    {'metric': 'mean_oracle_missing_count', 'value': float(per_sample_compare_df['oracle_missing_count'].mean())},
    {'metric': 'mean_last_missing_count', 'value': float(per_sample_compare_df['last_missing_count'].mean())},
    {'metric': 'samples_with_last_extra_reduction', 'value': int((per_sample_compare_df['last_extra_reduction_vs_raw'] > 0).sum())},
    {'metric': 'samples_with_last_missing_increase', 'value': int((per_sample_compare_df['last_missing_change_vs_raw'] > 0).sum())},
]
if 'prev_extra_count' in per_sample_compare_df.columns:
    summary_rows.extend([
        {'metric': 'mean_prev_extra_count', 'value': float(per_sample_compare_df['prev_extra_count'].mean())},
        {'metric': 'mean_prev_missing_count', 'value': float(per_sample_compare_df['prev_missing_count'].mean())},
    ])

presence_summary_df = pd.DataFrame(summary_rows)
display(presence_summary_df)
display(
    per_sample_compare_df.sort_values(
        ['last_extra_reduction_vs_raw', 'last_missing_change_vs_raw', 'raw_extra_count'],
        ascending=[False, True, False],
    ).head(20)
)

### [FASE 7] 13. Visualizacion cualitativa

In [ ]:
def show_sample(sample_id: str) -> None:
    image_gray = image_lookup[sample_id]
    raw_mask = raw_pred_lookup[sample_id]
    last_mask = last_pred_clip_lookup[sample_id]
    oracle_mask = oracle_clip_lookup[sample_id]
    target_mask = target_lookup[sample_id].copy()
    target_mask[target_mask == IGNORE_INDEX] = 0

    cols = 5 if sample_id in prev_range_clip_lookup else 4
    fig, axes = plt.subplots(1, cols, figsize=(4.4 * cols, 5))
    axes = np.atleast_1d(axes)
    axes[0].imshow(image_gray, cmap='gray')
    axes[0].set_title(f'ROI\n{sample_id}')
    axes[1].imshow(raw_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[1].set_title('Raw')
    offset = 2
    if sample_id in prev_range_clip_lookup:
        axes[2].imshow(prev_range_clip_lookup[sample_id], cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
        axes[2].set_title('Prev range clip')
        offset = 3
    axes[offset].imshow(last_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[offset].set_title('Last pred clip')
    axes[offset + 1].imshow(oracle_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[offset + 1].set_title('Oracle clip')
    axes[-1].imshow(target_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[-1].set_title('GT')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


sample_ids_to_show = per_sample_compare_df.sort_values(
    ['last_extra_reduction_vs_raw', 'last_missing_change_vs_raw', 'raw_extra_count'],
    ascending=[False, True, False],
)['unique_sample_id'].head(N_VIS_SAMPLES).tolist()

print('Muestras seleccionadas:', sample_ids_to_show)
for sample_id in sample_ids_to_show:
    show_sample(sample_id)

### [FASE 7] 14. Exportacion de resultados

In [ ]:
history_path = LAST_OUTPUT_DIR / 'last_visible_history.csv'
summary_path = LAST_OUTPUT_DIR / 'last_visible_summary.csv'
train_pred_path = LAST_OUTPUT_DIR / 'last_visible_train_predictions.csv'
val_pred_path = LAST_OUTPUT_DIR / 'last_visible_val_predictions.csv'
test_pred_path = LAST_OUTPUT_DIR / 'last_visible_test_predictions.csv'
compare_path = LAST_OUTPUT_DIR / 'last_visible_clipping_metric_comparison.csv'
raw_per_class_path = LAST_OUTPUT_DIR / 'last_visible_raw_per_class.csv'
oracle_per_class_path = LAST_OUTPUT_DIR / 'last_visible_oracle_per_class.csv'
last_per_class_path = LAST_OUTPUT_DIR / 'last_visible_pred_per_class.csv'
presence_path = LAST_OUTPUT_DIR / 'last_visible_presence_summary.csv'
per_sample_path = LAST_OUTPUT_DIR / 'last_visible_per_sample_compare.csv'

history_df.to_csv(history_path, index=False)
last_summary_df.to_csv(summary_path, index=False)
train_pred_df.to_csv(train_pred_path, index=False)
val_pred_df.to_csv(val_pred_path, index=False)
test_pred_df.to_csv(test_pred_path, index=False)
summary_compare_df.to_csv(compare_path, index=False)
raw_per_class_df.to_csv(raw_per_class_path, index=False)
oracle_per_class_df.to_csv(oracle_per_class_path, index=False)
last_per_class_df.to_csv(last_per_class_path, index=False)
presence_summary_df.to_csv(presence_path, index=False)
per_sample_compare_df.to_csv(per_sample_path, index=False)

experiment_summary_df = pd.DataFrame([
    {'metric': 'target_subset', 'value': MULTICLASS_SUBSET},
    {'metric': 'test_samples', 'value': int(len(test_frame))},
    {'metric': 'last_test_exact_acc', 'value': float(test_metrics['last_acc'])},
    {'metric': 'last_test_within1_acc', 'value': float(test_metrics['last_within1_acc'])},
    {'metric': 'last_test_mae', 'value': float(test_metrics['last_mae'])},
    {'metric': 'last_test_overprediction_rate', 'value': float(test_metrics['overprediction_rate'])},
    {'metric': 'raw_macro_dice_fg', 'value': float(summary_compare_df.loc[summary_compare_df['metric'] == 'macro_dice_fg', 'raw'].iloc[0])},
    {'metric': 'oracle_macro_dice_fg', 'value': float(summary_compare_df.loc[summary_compare_df['metric'] == 'macro_dice_fg', 'oracle_clip'].iloc[0])},
    {'metric': 'last_pred_clip_macro_dice_fg', 'value': float(summary_compare_df.loc[summary_compare_df['metric'] == 'macro_dice_fg', 'last_pred_clip'].iloc[0])},
    {'metric': 'mean_raw_extra_count', 'value': float(per_sample_compare_df['raw_extra_count'].mean())},
    {'metric': 'mean_last_extra_count', 'value': float(per_sample_compare_df['last_extra_count'].mean())},
    {'metric': 'mean_raw_missing_count', 'value': float(per_sample_compare_df['raw_missing_count'].mean())},
    {'metric': 'mean_last_missing_count', 'value': float(per_sample_compare_df['last_missing_count'].mean())},
])
experiment_path = LAST_OUTPUT_DIR / 'last_visible_experiment_summary.csv'
experiment_summary_df.to_csv(experiment_path, index=False)

display(experiment_summary_df)
print('Guardado:', history_path)
print('Guardado:', summary_path)
print('Guardado:', train_pred_path)
print('Guardado:', val_pred_path)
print('Guardado:', test_pred_path)
print('Guardado:', compare_path)
print('Guardado:', raw_per_class_path)
print('Guardado:', oracle_per_class_path)
print('Guardado:', last_per_class_path)
print('Guardado:', presence_path)
print('Guardado:', per_sample_path)
print('Guardado:', experiment_path)

> **Fase 7 (`_cuda`, notebook autocontenido, run completo):** cascada V3 (secc. 1–7) + **last_visible / clipping** (secc. 8) cerrados. Métricas cascada en `training_runs_cascade_v3_fase7_lastvis_cuda/`; auxiliar en `training_runs_last_visible_fase7_cuda/`.


### Análisis e interpretación

#### A. Cascada (secciones 1–7) — test

| Métrica | Fase 7 `_cuda` | Fase 4 `_cuda` (vivo) | Fase 6 `_cuda` (sin post) | Δ vs Fase 4 |
|---------|----------------|------------------------|---------------------------|-------------|
| Binario — Dice | **0,8710** | 0,8710 | 0,8710 | ≈0 |
| Binario — IoU | **0,7714** | 0,7714 | 0,7714 | ≈0 |
| `macro_dice_fg` | **0,2258** | **0,2380** | 0,2294 | **−0,0122** |
| `macro_iou_fg` | **0,1341** | **0,1415** | 0,1360 | **−0,0074** |
| `pixel_accuracy` | 0,7470 | 0,7502 | 0,7498 | −0,0032 |

**Mejor `val_macro_dice_fg`:** **0,2247** (época **22**). Referencia Fase 4: **0,2325** (ép. 22).

**Dice por clase (test)** — foco plan:

| Clase | Fase 7 | Fase 4 | Fase 6 |
|-------|--------|--------|--------|
| T9 | 0,0872 | 0,1324 | 0,1128 |
| T10 | 0,0849 | 0,0731 | 0,0777 |
| T11 | 0,1771 | 0,1750 | 0,1783 |
| L5 | 0,1871 | 0,1955 | 0,1925 |

T10 **sube** ligeramente vs Fase 4; **T9 y L5 bajan**; macro global **no** supera Fase 4.

#### B. Last visible (sección 8) — test (`last_visible_summary.csv`, n=40)

| Métrica | Valor |
|---------|-------|
| Exactitud índice última vértebra | **0,25** |
| Within-1 | **0,425** |
| MAE (índices) | **2,025** |
| Tasa sobrepredicción (rango) | **0,525** |
| Tasa subpredicción | 0,225 |

El estimador **no** alcanza utilidad clínica operativa (exactitud baja; MAE alto). Val/ train: within-1 ~0,69–0,84, pero **test** queda muy por debajo → generalización insuficiente.

#### C. Clipping multiclase — test (`last_visible_clipping_metric_comparison.csv`)

| Modo | `macro_dice_fg` | `macro_iou_fg` | Nota |
|------|-----------------|----------------|------|
| `raw` (sin clip) | 0,2216 | 0,1315 | Predicción cascada de este run |
| `last_pred_clip` | **0,2295** | 0,1367 | Clip con índice estimado |
| `oracle_clip` | 0,2491 | 0,1508 | Techo con GT last_visible |
| `prev_range_clip` | 0,2827 | 0,1737 | Referencia histórica (CSV legacy); **no** comparable a adopción |

**Efecto conteo vértebras** (`last_visible_presence_summary.csv`): media vértebras extra **2,48 → 1,35** con `last_pred_clip`, pero faltantes **0,65 → 1,38** (9/40 muestras empeoran missing). Mejora de macro **+0,0079** vs `raw` del mismo run, pero **sigue por debajo** de Fase 4 sin clipping (0,2380).

#### Lectura frente al plan (§3 Fase 7)

1. **Cascada autocontenida** reproduce binario alineado al vivo pero **no** iguala ni supera Fase 4 en `macro_dice_fg` test.
2. **Last visible:** métricas de test insuficientes; no justifica integrar el auxiliar en el pipeline.
3. **Clipping `last_pred_clip`:** mejora marginal local frente a `raw` del mismo run; **no** compensa la brecha vs Fase 4 ni el coste en vértebras faltantes.

**Decisión de fase:** **No adoptar** — mantener pipeline vivo **Fase 3 + Fase 4**; **no** sustituir por checkpoints fase7 ni integrar `last_visible` / clipping en producción con este run.



### Sugerencias posteriores

1. **Fase 8** (eficiencia / inferencia) según `PLAN_ACCION_AJUSTES_MODELOS.md` §3.
2. Si se reabre last_visible: más datos, features o arquitectura; objetivo mínimo **within-1 test > 0,7** antes de reevaluar clipping.
3. Explorar clipping **solo** si el estimador mejora; `oracle_clip` muestra techo (~0,249) pero no es desplegable.
4. Tras cada run, ejecutar la celda de **Registro de ejecución** al pie del notebook.


## Registro de ejecución del notebook

La **fecha de última modificación** del archivo `.ipynb` en disco **no** indica cuándo terminó el entrenamiento: cambia en cada guardado.

**Recomendación:** al finalizar entrenamiento y evaluación, ejecutar la **celda de código siguiente** (la salida queda guardada en el notebook y marca el momento de esa ejecución). Opcionalmente completar la tabla.

| Campo | Valor |
|-------|--------|
| **Fecha de cierre del run** | *(ver salida de la celda siguiente o anotar)* |
| **Entorno** | *(p. ej. Colab T4, VS Code + CUDA, …)* |


In [ ]:
# --- Registro de ejecución: ejecutar al finalizar entrenamiento y evaluación ---
from __future__ import annotations

from datetime import datetime, timezone

now_local = datetime.now().astimezone()
now_utc = datetime.now(timezone.utc)
print("Ejecución de esta celda — local:", now_local.strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Ejecución de esta celda — UTC:  ", now_utc.strftime("%Y-%m-%d %H:%M:%S UTC"))
